In [1]:

import os
import glob
import csv
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# DAY 20: FINAL MANTLE CONVECTION PROJECT
# --------------------------------------------------

# Locate the simulation results
folders = glob.glob("combined_mantle_results*")

if not folders:
    raise FileNotFoundError(
        "Cannot find combined_mantle_results folder. "
        "Run the simulation first."
    )

result_folder = max(folders, key=os.path.getmtime)

files = glob.glob(os.path.join(result_folder, "*.npz"))

if not files:
    raise FileNotFoundError(
        "No .npz simulation files found."
    )

output_folder = "day20_final_project"
os.makedirs(output_folder, exist_ok=True)

datasets = []

# --------------------------------------------------
# 1. LOAD AND CHECK DATA
# --------------------------------------------------

for filename in files:

    with np.load(filename) as data:

        required = ["ra", "x", "z", "T", "u", "w"]

        if not all(key in data for key in required):
            print("Skipping incomplete file:", filename)
            continue

        Ra = float(np.asarray(data["ra"]).item())

        x = data["x"].copy()
        z = data["z"].copy()
        T = data["T"].copy()
        u = data["u"].copy()
        w = data["w"].copy()

    if (
        T.ndim != 2
        or T.shape != (len(z), len(x))
        or u.shape != T.shape
        or w.shape != T.shape
    ):
        print("Skipping incompatible arrays:", filename)
        continue

    if not all(
        np.all(np.isfinite(a))
        for a in [x, z, T, u, w]
    ):
        print("Skipping non-finite data:", filename)
        continue

    if len(x) < 3 or len(z) < 3:
        print("Skipping grid that is too small:", filename)
        continue

    dx = float(np.mean(np.diff(x)))
    dz = float(np.mean(np.diff(z)))

    if dx <= 0 or dz <= 0:
        print("Skipping invalid grid:", filename)
        continue

    # ----------------------------------------------
    # TEMPERATURE
    # ----------------------------------------------

    speed = np.sqrt(u**2 + w**2)

    T_mean = np.mean(T, axis=1)
    T_conduction = 1.0 - z

    temperature_anomaly = T_mean - T_conduction

    rms_temperature_anomaly = np.sqrt(
        np.mean(temperature_anomaly**2)
    )

    # ----------------------------------------------
    # VELOCITY
    # ----------------------------------------------

    max_speed = np.max(speed)

    rms_speed = np.sqrt(
        np.mean(speed[1:-1, 1:-1]**2)
    )

    # ----------------------------------------------
    # NUSSELT NUMBER
    # ----------------------------------------------

    Nu_bottom = -np.mean(
        (T[1, 1:-1] - T[0, 1:-1]) / dz
    )

    Nu_top = -np.mean(
        (T[-1, 1:-1] - T[-2, 1:-1]) / dz
    )

    denominator = max(
        abs(Nu_bottom),
        abs(Nu_top),
        1e-12
    )

    mismatch = (
        abs(Nu_bottom - Nu_top) / denominator
    ) * 100

    # ----------------------------------------------
    # TEMPERATURE-FLOW CORRELATION
    # ----------------------------------------------

    T_anomaly = T - np.mean(T, axis=1, keepdims=True)

    Ta = T_anomaly[1:-1, 1:-1]
    wa = w[1:-1, 1:-1]

    if np.std(Ta) > 1e-12 and np.std(wa) > 1e-12:
        correlation = np.corrcoef(
            Ta.ravel(), wa.ravel()
        )[0, 1]
    else:
        correlation = np.nan

    # ----------------------------------------------
    # SAVE RESULTS
    # ----------------------------------------------

    datasets.append({
        "Ra": Ra,
        "x": x,
        "z": z,
        "T": T,
        "u": u,
        "w": w,
        "speed": speed,
        "max_speed": max_speed,
        "rms_speed": rms_speed,
        "Nu_bottom": Nu_bottom,
        "Nu_top": Nu_top,
        "mismatch": mismatch,
        "Tmin": np.min(T),
        "Tmax": np.max(T),
        "rms_anomaly": rms_temperature_anomaly,
        "correlation": correlation
    })

if not datasets:
    raise ValueError(
        "No valid simulation datasets were found."
    )

datasets.sort(key=lambda item: item["Ra"])

print("Number of valid simulations:", len(datasets))

# --------------------------------------------------
# 2. CREATE FINAL SUMMARY CSV
# --------------------------------------------------

csv_path = os.path.join(
    output_folder,
    "day20_final_summary.csv"
)

headers = [
    "Rayleigh_number",
    "Maximum_speed",
    "RMS_interior_speed",
    "Nu_bottom",
    "Nu_top",
    "Flux_mismatch_percent",
    "Minimum_temperature",
    "Maximum_temperature",
    "RMS_temperature_departure",
    "Temperature_vertical_velocity_correlation"
]

with open(csv_path, "w", newline="") as file:

    writer = csv.writer(file)
    writer.writerow(headers)

    for d in datasets:
        writer.writerow([
            d["Ra"],
            d["max_speed"],
            d["rms_speed"],
            d["Nu_bottom"],
            d["Nu_top"],
            d["mismatch"],
            d["Tmin"],
            d["Tmax"],
            d["rms_anomaly"],
            d["correlation"]
        ])

# --------------------------------------------------
# 3. FINAL TEMPERATURE FIELDS
# --------------------------------------------------

n = len(datasets)

fig, axes = plt.subplots(
    1, n,
    figsize=(5 * n, 4),
    squeeze=False
)

for i, d in enumerate(datasets):

    contour = axes[0, i].contourf(
        d["x"], d["z"], d["T"],
        levels=25,
        cmap="inferno"
    )

    axes[0, i].set_title(f"Ra = {d['Ra']:g}")
    axes[0, i].set_xlabel("Horizontal position, x")
    axes[0, i].set_ylabel("Height, z")

fig.colorbar(
    contour,
    ax=axes.ravel().tolist(),
    label="Normalized temperature",
    pad=0.03
)

fig.suptitle("Final Results: Temperature Fields")

fig.subplots_adjust(
    top=0.82,
    bottom=0.15,
    left=0.08,
    right=0.90,
    wspace=0.35
)

fig.savefig(
    os.path.join(output_folder, "final_temperature_fields.png"),
    dpi=200,
    bbox_inches="tight"
)

plt.close(fig)

# --------------------------------------------------
# 4. FINAL VELOCITY FIELDS
# --------------------------------------------------

fig, axes = plt.subplots(
    1, n,
    figsize=(5 * n, 4),
    squeeze=False
)

for i, d in enumerate(datasets):

    contour = axes[0, i].contourf(
        d["x"], d["z"], d["speed"],
        levels=25,
        cmap="viridis"
    )

    axes[0, i].set_title(f"Ra = {d['Ra']:g}")
    axes[0, i].set_xlabel("Horizontal position, x")
    axes[0, i].set_ylabel("Height, z")

fig.colorbar(
    contour,
    ax=axes.ravel().tolist(),
    label="Velocity magnitude",
    pad=0.03
)

fig.suptitle("Final Results: Velocity Fields")

fig.subplots_adjust(
    top=0.82,
    bottom=0.15,
    left=0.08,
    right=0.90,
    wspace=0.35
)

fig.savefig(
    os.path.join(output_folder, "final_velocity_fields.png"),
    dpi=200,
    bbox_inches="tight"
)

plt.close(fig)

# --------------------------------------------------
# 5. VELOCITY VERSUS RAYLEIGH NUMBER
# --------------------------------------------------

Ra_values = [d["Ra"] for d in datasets]
max_speeds = [d["max_speed"] for d in datasets]
rms_speeds = [d["rms_speed"] for d in datasets]

plt.figure(figsize=(7, 5))

plt.plot(
    Ra_values, max_speeds,
    "o-", label="Maximum speed"
)

plt.plot(
    Ra_values, rms_speeds,
    "s--", label="RMS interior speed"
)

plt.xlabel("Rayleigh number")
plt.ylabel("Velocity magnitude")
plt.title("Final Results: Flow Strength")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(output_folder, "final_velocity_vs_Ra.png"),
    dpi=200
)

plt.close()

# --------------------------------------------------
# 6. NUSSELT NUMBER VERSUS RAYLEIGH NUMBER
# --------------------------------------------------

Nu_bottom = [d["Nu_bottom"] for d in datasets]
Nu_top = [d["Nu_top"] for d in datasets]

plt.figure(figsize=(7, 5))

plt.plot(
    Ra_values, Nu_bottom,
    "o-", label="Bottom Nu"
)

plt.plot(
    Ra_values, Nu_top,
    "s--", label="Top Nu"
)

plt.axhline(
    1.0, linestyle=":",
    label="Conductive reference"
)

plt.xlabel("Rayleigh number")
plt.ylabel("Nusselt number")
plt.title("Final Results: Heat Transfer")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(output_folder, "final_Nu_vs_Ra.png"),
    dpi=200
)

plt.close()

# --------------------------------------------------
# 7. GENERATE FINAL REPORT DRAFT
# --------------------------------------------------

report_path = os.path.join(
    output_folder,
    "day20_final_report_draft.txt"
)

with open(report_path, "w", encoding="utf-8") as file:

    file.write(
        "2-D NUMERICAL MODELLING OF THERMAL CONVECTION "
        "IN THE EARTH'S MANTLE\n"
    )

    file.write("=" * 65 + "\n\n")

    file.write("1. OBJECTIVE\n")
    file.write(
        "To investigate temperature and flow patterns in a "
        "simplified 2-D thermal convection model and compare "
        "diagnostics across Rayleigh numbers.\n\n"
    )

    file.write("2. THEORETICAL BACKGROUND\n")
    file.write(
        "The Rayleigh number measures the relative importance "
        "of buoyancy-driven transport compared with viscous and "
        "thermal-diffusion effects. The Nusselt number compares "
        "heat transfer with a conductive reference state.\n\n"
    )

    file.write("3. NUMERICAL MODEL\n")
    file.write(
        "The model uses a 2-D rectangular domain with a hot "
        "bottom and cold top. The saved temperature and velocity "
        "fields are analysed using Python, NumPy, and Matplotlib. "
        "The precise flow formulation and numerical assumptions "
        "must be documented from the code actually used.\n\n"
    )

    file.write("4. RESULTS SUMMARY\n")

    for d in datasets:
        file.write(
            f"\nRa = {d['Ra']:g}\n"
            f"  Maximum speed = {d['max_speed']:.6g}\n"
            f"  RMS interior speed = {d['rms_speed']:.6g}\n"
            f"  Bottom Nu = {d['Nu_bottom']:.6g}\n"
            f"  Top Nu = {d['Nu_top']:.6g}\n"
            f"  Flux mismatch = {d['mismatch']:.4f}%\n"
            f"  Temperature range = {d['Tmin']:.6g} "
            f"to {d['Tmax']:.6g}\n"
            f"  RMS departure from conduction = "
            f"{d['rms_anomaly']:.6g}\n"
        )

    file.write("\n5. DISCUSSION\n")
    file.write(
        "Compare the calculated velocity, temperature fields, "
        "and Nusselt numbers across Rayleigh numbers. Discuss "
        "the observed trends using the actual plots and tables. "
        "Do not assume that all diagnostics increase "
        "monotonically with Rayleigh number.\n\n"
    )

    file.write("6. LIMITATIONS\n")
    file.write(
        "The model is a simplified representation of mantle "
        "convection. Numerical convergence, time-step sensitivity, "
        "boundary conditions, heat-flux consistency, and the "
        "accuracy of the flow formulation limit the conclusions "
        "that can be drawn. Saved snapshots do not by themselves "
        "establish steady state or full energy conservation.\n\n"
    )

    file.write("7. CONCLUSION\n")
    file.write(
        "The simulations provide a framework for examining "
        "the influence of Rayleigh number on temperature and "
        "flow in a simplified 2-D model. Final conclusions "
        "should be stated only after reviewing the numerical "
        "results and their validation diagnostics.\n"
    )

# --------------------------------------------------
# 8. FINAL STATUS
# --------------------------------------------------

print("\n" + "=" * 50)
print("DAY 20 FINAL PROJECT OUTPUTS CREATED")
print("=" * 50)

print("Input folder:", result_folder)
print("Output folder:", output_folder)

print("\nFiles created:")
print("1. day20_final_summary.csv")
print("2. final_temperature_fields.png")
print("3. final_velocity_fields.png")
print("4. final_velocity_vs_Ra.png")
print("5. final_Nu_vs_Ra.png")
print("6. day20_final_report_draft.txt")

print("\nReview all outputs before submitting the project.")


Number of valid simulations: 4

DAY 20 FINAL PROJECT OUTPUTS CREATED
Input folder: combined_mantle_results_2
Output folder: day20_final_project

Files created:
1. day20_final_summary.csv
2. final_temperature_fields.png
3. final_velocity_fields.png
4. final_velocity_vs_Ra.png
5. final_Nu_vs_Ra.png
6. day20_final_report_draft.txt

Review all outputs before submitting the project.
